# ELEN4025 – Machine Learning Group Project
## Stage 1: Data Loading & Sanity Checks

In [ ]:
pip install pandas numpy

Note: you may need to restart the kernel to use updated packages.



[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
#Set Up
import pandas as pd
import numpy as np
import os
import subprocess
import zipfile

RAW_DIR = os.path.join("data", "raw")
PROCESSED_DIR = os.path.join("data", "processed")
os.makedirs(RAW_DIR, exist_ok=True)
os.makedirs(PROCESSED_DIR, exist_ok=True)

EXPECTED_TABLES = {
    "studentInfo":         {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "studentVle":          {"min_rows": 10000000, "key_cols": ["code_module", "code_presentation", "id_student", "id_site", "date"]},
    "assessments":         {"min_rows": 200, "key_cols": ["code_module", "code_presentation", "id_assessment"]},
    "studentAssessment":   {"min_rows": 170000, "key_cols": ["id_assessment", "id_student"]},
    "studentRegistration": {"min_rows": 30000, "key_cols": ["code_module", "code_presentation", "id_student"]},
    "courses":             {"min_rows": 20, "key_cols": ["code_module", "code_presentation"]},
    "vle":                 {"min_rows": 6000, "key_cols": ["id_site", "code_module", "code_presentation"]},
}

print("Configuration ready.")
print(f"  RAW_DIR:       {RAW_DIR}")
print(f"  PROCESSED_DIR: {PROCESSED_DIR}")
print(f"  Expected tables: {list(EXPECTED_TABLES.keys())}")

Configuration ready.
  RAW_DIR:       data\raw
  PROCESSED_DIR: data\processed
  Expected tables: ['studentInfo', 'studentVle', 'assessments', 'studentAssessment', 'studentRegistration', 'courses', 'vle']


In [ ]:
#Download and Extract Dataset 
ZIP_PATH = os.path.join("data", "oulad.zip")

if os.path.exists(ZIP_PATH):
    with zipfile.ZipFile(ZIP_PATH, "r") as zf:
        zf.extractall(RAW_DIR)
        extracted = zf.namelist()
    print(f"Extracted {len(extracted)} files from {ZIP_PATH}:")
    for f in sorted(extracted):
        print(f"  {f}")
else:
    print(f"Zip file not found at {ZIP_PATH}.")
    print("Checking if CSVs already exist in data/raw/ ...")
    existing = [f for f in os.listdir(RAW_DIR) if f.endswith(".csv")]
    print(f"  Found {len(existing)} CSV files: {existing}")

Extracted 7 files from data\oulad.zip:
  assessments.csv
  courses.csv
  studentAssessment.csv
  studentInfo.csv
  studentRegistration.csv
  studentVle.csv
  vle.csv


In [ ]:
#Check all CSV files have been exraccted
expected_files = [f"{name}.csv" for name in EXPECTED_TABLES]
for fname in expected_files:
    path = os.path.join(RAW_DIR, fname)
    assert os.path.exists(path), f"FAIL: Missing file {path}"
    size_mb = os.path.getsize(path) / 1e6
    print(f"   {fname:30s} ({size_mb:.1f} MB)")

print(f"\n All {len(expected_files)} CSV files present.")

   studentInfo.csv                (3.5 MB)
   studentVle.csv                 (453.8 MB)
   assessments.csv                (0.0 MB)
   studentAssessment.csv          (5.7 MB)
   studentRegistration.csv        (1.1 MB)
   courses.csv                    (0.0 MB)
   vle.csv                        (0.3 MB)

 All 7 CSV files present.


In [ ]:
# Load CSV Tables 
tables = {}
for name in EXPECTED_TABLES:
    path = os.path.join(RAW_DIR, f"{name}.csv")
    tables[name] = pd.read_csv(path)
    print(f"Loaded {name:25s} -> {tables[name].shape[0]:>10,} rows x {tables[name].shape[1]:>3} cols")

print(f"\nAll {len(tables)} tables loaded.")

Loaded studentInfo               ->     32,593 rows x  12 cols
Loaded studentVle                -> 10,655,280 rows x   6 cols
Loaded assessments               ->        206 rows x   6 cols
Loaded studentAssessment         ->    173,912 rows x   5 cols
Loaded studentRegistration       ->     32,593 rows x   5 cols
Loaded courses                   ->         22 rows x   3 cols
Loaded vle                       ->      6,364 rows x   6 cols

All 7 tables loaded.


In [ ]:
#Check and confirm all tables have been loaded correctly 
assert len(tables) == 7, f"FAIL: Expected 7 tables, got {len(tables)}"
print("All 7 tables loaded into memory.")

for name, df in tables.items():
    assert len(df) > 0, f"FAIL: {name} is empty"
print("No empty tables.")

for name, df in tables.items():
    assert df.shape[0] >= EXPECTED_TABLES[name]["min_rows"], (
        f"FAIL: {name} has {df.shape[0]} rows, expected >= {EXPECTED_TABLES[name]['min_rows']}"
    )
print("All row counts in expected range.")



All 7 tables loaded into memory.
No empty tables.
All row counts in expected range.


In [ ]:
#Inspect Data Types and Shapes
for name, df in tables.items():
    print(f"\n{'─'*55}")
    print(f"  {name}  |  {df.shape[0]:,} rows x {df.shape[1]} cols")
    print(f"{'─'*55}")
    print(df.dtypes.to_string())
    print(f"\nFirst 3 rows:")
    print(df.head(3).to_string())


───────────────────────────────────────────────────────
  studentInfo  |  32,593 rows x 12 cols
───────────────────────────────────────────────────────
code_module               str
code_presentation         str
id_student              int64
gender                    str
region                    str
highest_education         str
imd_band                  str
age_band                  str
num_of_prev_attempts    int64
studied_credits         int64
disability                str
final_result              str

First 3 rows:
  code_module code_presentation  id_student gender                region      highest_education imd_band age_band  num_of_prev_attempts  studied_credits disability final_result
0         AAA             2013J       11391      M   East Anglian Region       HE Qualification  90-100%     55<=                     0              240          N         Pass
1         AAA             2013J       28400      F              Scotland       HE Qualification   20-30%    35-55     

In [ ]:
#Verify Key Columns Exist with correct types 
for name, df in tables.items():
    for col in EXPECTED_TABLES[name]["key_cols"]:
        assert col in df.columns, f"FAIL: {name} missing column '{col}'"
print("All key columns present in every table.")

# Check specific expected column counts
assert tables["studentInfo"].shape[1] == 12, "FAIL: studentInfo should have 12 columns"
assert tables["studentVle"].shape[1] == 6, "FAIL: studentVle should have 6 columns"
assert tables["courses"].shape[1] == 3, "FAIL: courses should have 3 columns"
assert tables["assessments"].shape[1] == 6, "FAIL: assessments should have 6 columns"
assert tables["studentAssessment"].shape[1] == 5, "FAIL: studentAssessment should have 5 columns"
assert tables["studentRegistration"].shape[1] == 5, "FAIL: studentRegistration should have 5 columns"
assert tables["vle"].shape[1] == 6, "FAIL: vle should have 6 columns"
print(" All Column counts correct for all tables.")

# sum_click must be numeric for aggregation; id_student must be numeric for joins.
# If pandas read them as strings, downstream computations would silently fail.
assert tables["studentVle"]["sum_click"].dtype in [np.int64, np.int32], "FAIL: sum_click should be integer"
assert tables["studentInfo"]["id_student"].dtype in [np.int64, np.int32], "FAIL: id_student should be integer"
print("Critical columns have expected dtypes.")



All key columns present in every table.
 All Column counts correct for all tables.
Critical columns have expected dtypes.


## Missing/ Null Vlue Analysis 

In [ ]:
#Identify Missing Data 
print("Missing value analysis per table:\n")

missing_summary = []
for name, df in tables.items():
    total_missing = df.isnull().sum().sum()
    if total_missing == 0:
        print(f"  {name}: ✓ no missing values")
    else:
        miss = df.isnull().sum()
        miss = miss[miss > 0]
        for col, count in miss.items():
            pct = 100.0 * count / len(df)
            missing_summary.append({
                "Table": name,
                "Column": col,
                "Missing Count": count,
                "Missing %": round(pct, 2),
            })
            print(f"  {name}.{col}: {count:,} missing ({pct:.2f}%)")

Missing value analysis per table:

  studentInfo.imd_band: 1,111 missing (3.41%)
  studentVle: ✓ no missing values
  assessments.date: 11 missing (5.34%)
  studentAssessment.score: 173 missing (0.10%)
  studentRegistration.date_registration: 45 missing (0.14%)
  studentRegistration.date_unregistration: 22,521 missing (69.10%)
  courses: ✓ no missing values
  vle.week_from: 5,243 missing (82.39%)
  vle.week_to: 5,243 missing (82.39%)


In [ ]:
#Table shwoing missing values summary
missing_df = pd.DataFrame(missing_summary)
print("\n─── Missing Values Summary Table ───\n")
print(missing_df.to_string(index=False))


─── Missing Values Summary Table ───

              Table              Column  Missing Count  Missing %
        studentInfo            imd_band           1111       3.41
        assessments                date             11       5.34
  studentAssessment               score            173       0.10
studentRegistration   date_registration             45       0.14
studentRegistration date_unregistration          22521      69.10
                vle           week_from           5243      82.39
                vle             week_to           5243      82.39


In [ ]:
#Document Missing Values and Confirm Critical Columns are Clean
si = tables["studentInfo"]

for col in ["id_student", "final_result", "code_module", "code_presentation", "gender", "region"]:
    assert si[col].isnull().sum() == 0, f"FAIL: studentInfo.{col} has nulls"
print("No missing values in critical studentInfo columns.")


assert tables["studentVle"].isnull().sum().sum() == 0, "FAIL: studentVle has nulls"
print("studentVle has zero missing values.")


assert tables["courses"].isnull().sum().sum() == 0, "FAIL: courses has nulls"
print("courses has zero missing values.")


assert si["imd_band"].isnull().sum() == 1111, "FAIL: imd_band missing count unexpected"
print("imd_band: 1,111 missing.")

unreg_missing = tables["studentRegistration"]["date_unregistration"].isnull().sum()
assert unreg_missing == 22521, f"FAIL: date_unregistration missing count unexpected: {unreg_missing}"
print(f"date_unregistration: {unreg_missing:,} missing.")

score_missing = tables["studentAssessment"]["score"].isnull().sum()
assert score_missing == 173, f"FAIL: score missing count unexpected: {score_missing}"
print(f"score: {score_missing} missing.")

print("\n All missing values documented.")

No missing values in critical studentInfo columns.
studentVle has zero missing values.
courses has zero missing values.
imd_band: 1,111 missing.
date_unregistration: 22,521 missing.
score: 173 missing.

 All missing values documented.


In [ ]:
#Identify and remove duplicate rows 
print("Duplicate analysis per table:\n")
print(f"{'Table':25s} {'Duplicates':>12s}   {'Decision':30s}")
print("─" * 75)

for name, df in tables.items():
    n_dup = df.duplicated().sum()
    if name == "studentVle" and n_dup > 0:
        decision = "KEEP"
    elif n_dup > 0:
        decision = "REMOVE"
    else:
        decision = "OK — none found"
    print(f"{name:25s} {n_dup:>12,}   {decision}")


Duplicate analysis per table:

Table                       Duplicates   Decision                      
───────────────────────────────────────────────────────────────────────────
studentInfo                          0   OK — none found
studentVle                     787,170   KEEP
assessments                          0   OK — none found
studentAssessment                    0   OK — none found
studentRegistration                  0   OK — none found
courses                              0   OK — none found
vle                                  0   OK — none found


### Duplicate Handling

`studentVle` duplicates are kept because they represent valid repeated clicks on the same resource on the same day. Removing them would undercount real VLE engagement.

All other tables have zero duplicates, so no duplicate removal was required.

In [ ]:
#Raw Distrabution of final Results
si = tables["studentInfo"].copy()

print("Raw final_result distribution:\n")
counts = si["final_result"].value_counts()
for val, count in counts.items():
    pct = 100 * count / len(si)
    print(f"  {val:15s}: {count:>6,}  ({pct:.1f}%)")
print(f"\n  Total registrations: {len(si):,}")


Raw final_result distribution:

  Pass           : 12,361  (37.9%)
  Withdrawn      : 10,156  (31.2%)
  Fail           :  7,052  (21.6%)
  Distinction    :  3,024  (9.3%)

  Total registrations: 32,593


In [ ]:
# Note: "Withdrawn" is the second-largest group (31.2%).
# These are students who de-registered before completing the course.
# Treating them as "unfavourable" is appropriate: from an early-warning
# perspective, we want to flag students at risk of EITHER failing or dropping out.

### Step 6b — Map to binary target

The target is **unfavourable vs favourable outcome**:

| Original `final_result` | Binary `target` | Category |
|---|---|---|
| Pass | 1 | Favourable |
| Distinction | 1 | Favourable |
| Fail | 0 | Unfavourable |
| Withdrawn | 0 | Unfavourable |

**Design decision:** We group `Withdrawn` with `Fail` because the university's
goal is early intervention: identifying students who might either fail or
drop out so that support can be offered. A student who withdraws is just as
much a "lost" outcome as one who fails.

In [ ]:
LABEL_MAP = {
    "Pass": 1,         # Favourable — completed and passed
    "Distinction": 1,  # Favourable — completed with distinction
    "Fail": 0,         # Unfavourable — completed but did not pass
    "Withdrawn": 0,    # Unfavourable — dropped out before completion
}

si["target"] = si["final_result"].map(LABEL_MAP)

target_counts = si["target"].value_counts().sort_index()
print("Binary target distribution:\n")
for label, count in target_counts.items():
    pct = 100 * count / len(si)
    tag = "favourable (Pass/Distinction)" if label == 1 else "unfavourable (Fail/Withdrawn)"
    print(f"  target = {label} — {tag}: {count:>6,}  ({pct:.1f}%)")

class_ratio = target_counts[1] / target_counts[0]
print(f"\nClass ratio (favourable / unfavourable): {class_ratio:.3f}")
print(f"Imbalance: {'Moderate — no resampling strictly needed' if 0.5 < class_ratio < 2.0 else 'Significant — consider resampling'}")

tables["studentInfo"] = si

Binary target distribution:

  target = 0 — unfavourable (Fail/Withdrawn): 17,208  (52.8%)
  target = 1 — favourable (Pass/Distinction): 15,385  (47.2%)

Class ratio (favourable / unfavourable): 0.894
Imbalance: Moderate — no resampling strictly needed


In [ ]:
#Check Binary Target is valid
# No unmapped values — if final_result contained a category we didn't anticipate,
# map() would produce NaN, which would silently corrupt all model training.
assert si["target"].isnull().sum() == 0, "FAIL: Some final_result values were not mapped!"
print("All final_result values successfully mapped (no nulls in target).")

# Only {0, 1} — multi-class labels would break the binary classifiers
assert set(si["target"].unique()) == {0, 1}, "FAIL: Target is not binary {0, 1}!"
print("Target is strictly binary: {0, 1}.")

# Verify each mapping individually to catch accidental swaps
assert (si.loc[si["final_result"] == "Pass", "target"] == 1).all(), "FAIL: Pass != 1"
assert (si.loc[si["final_result"] == "Distinction", "target"] == 1).all(), "FAIL: Distinction != 1"
assert (si.loc[si["final_result"] == "Fail", "target"] == 0).all(), "FAIL: Fail != 0"
assert (si.loc[si["final_result"] == "Withdrawn", "target"] == 0).all(), "FAIL: Withdrawn != 0"
print("Mapping verified: Pass→1, Distinction→1, Fail→0, Withdrawn→0.")

# Row count must be unchanged — mapping should not add or drop any rows
assert len(si) == 32593, f"FAIL: Expected 32,593 rows, got {len(si)}"
print("Row count unchanged: 32,593.")

# Class balance check — if either class drops below 10%, standard metrics
# like accuracy become misleading and we'd need to rely on F1/AUC exclusively.
min_pct = 100 * target_counts.min() / len(si)
assert min_pct > 10, f"FAIL: Minority class only {min_pct:.1f}%"
print(f"Minority class = {min_pct:.1f}% — moderate imbalance, workable with stratified splits.")

# counts must add up to total rows
assert target_counts.sum() == len(si), "FAIL: Target counts don't sum to total rows"
print(f"Target counts sum to total: {target_counts.sum():,}.")

print("\n Binary target created and validated.")

All final_result values successfully mapped (no nulls in target).
Target is strictly binary: {0, 1}.
Mapping verified: Pass→1, Distinction→1, Fail→0, Withdrawn→0.
Row count unchanged: 32,593.
Minority class = 47.2% — moderate imbalance, workable with stratified splits.
Target counts sum to total: 32,593.

 Binary target created and validated.


In [ ]:
#Referential Integrity Checks
# The composite key (code_module, code_presentation, id_student) uniquely
# identifies each student-course registration. All child tables must reference
# only keys that exist in studentInfo.
si_keys = set(zip(si["code_module"], si["code_presentation"], si["id_student"]))

# Check 1: Every student in the click-log must exist in studentInfo,
# otherwise their engagement features can't be linked to an outcome.
svle = tables["studentVle"]
svle_keys = set(zip(svle["code_module"], svle["code_presentation"], svle["id_student"]))
orphan_vle = svle_keys - si_keys

# Check 2: Registration records must match. mismatches would indicate
# students who registered but have no demographic/outcome data.
sreg = tables["studentRegistration"]
sreg_keys = set(zip(sreg["code_module"], sreg["code_presentation"], sreg["id_student"]))
orphan_reg = sreg_keys - si_keys

# Check 3: Assessment IDs in student submissions must reference valid assessments.
# Invalid IDs would mean scores can't be tied to their assessment type/weight/date.
assess_ids = set(tables["assessments"]["id_assessment"])
sa_assess_ids = set(tables["studentAssessment"]["id_assessment"])
orphan_sa_ids = sa_assess_ids - assess_ids

# Check 4: Students who submitted assessments must exist in studentInfo.
# We need to join through assessments first to get the module/presentation context.
sa_merged = tables["studentAssessment"].merge(
    tables["assessments"][["id_assessment", "code_module", "code_presentation"]],
    on="id_assessment", how="left"
)
sa_keys = set(zip(sa_merged["code_module"], sa_merged["code_presentation"], sa_merged["id_student"]))
orphan_sa_stu = sa_keys - si_keys

print("Referential Integrity Checks:\n")
print(f"  studentVle students NOT in studentInfo:            {len(orphan_vle)}")
print(f"  studentRegistration students NOT in studentInfo:   {len(orphan_reg)}")
print(f"  studentAssessment assessment IDs NOT in assessments: {len(orphan_sa_ids)}")
print(f"  studentAssessment students NOT in studentInfo:     {len(orphan_sa_stu)}")

assert len(orphan_vle) == 0
assert len(orphan_reg) == 0
assert len(orphan_sa_ids) == 0
assert len(orphan_sa_stu) == 0
print("\n All join keys are consistent.")




Referential Integrity Checks:

  studentVle students NOT in studentInfo:            0
  studentRegistration students NOT in studentInfo:   0
  studentAssessment assessment IDs NOT in assessments: 0
  studentAssessment students NOT in studentInfo:     0

 All join keys are consistent.


In [ ]:
#Full summary Table 
summary_rows = []
for name, df in tables.items():
    summary_rows.append({
        "Table": name,
        "Rows": df.shape[0],
        "Columns": df.shape[1],
        "Missing Cells": df.isnull().sum().sum(),
        "Duplicate Rows": df.duplicated().sum(),
        "Memory (MB)": round(df.memory_usage(deep=True).sum() / 1e6, 1),
    })

summary_table = pd.DataFrame(summary_rows)
print("─── Sanity-Check Summary Table ───\n")
print(summary_table.to_string(index=False))

─── Sanity-Check Summary Table ───

              Table     Rows  Columns  Missing Cells  Duplicate Rows  Memory (MB)
        studentInfo    32593       13           1111               0         17.3
         studentVle 10655280        6              0          787170       1470.4
        assessments      206        6             11               0          0.0
  studentAssessment   173912        5            173               0          7.0
studentRegistration    32593        5          22566               0          4.2
            courses       22        3              0               0          0.0
                vle     6364        6          10486               0          1.2


In [ ]:
# Save processed output locally
# Note: data/processed/ is ignored by Git, so this file is regenerated when the notebook is run.
out_path = os.path.join(PROCESSED_DIR, "studentInfo_with_target.csv")

si.to_csv(out_path, index=False)

print(f"Saved locally: {out_path}")
print(f"{len(si):,} rows, {si.shape[1]} columns")

Saved locally: data\processed\studentInfo_with_target.csv
32,593 rows, 13 columns


In [ ]:
# Verify saved output
si_check = pd.read_csv(out_path)

assert si_check.shape == si.shape, f"FAIL: Saved shape {si_check.shape} != {si.shape}"
assert "target" in si_check.columns, "FAIL: 'target' column missing"
assert si_check["target"].isnull().sum() == 0, "FAIL: target has nulls"
assert set(si_check["target"].unique()) == {0, 1}, "FAIL: target not {0,1}"

saved_counts = si_check["target"].value_counts().sort_index()
assert (saved_counts == target_counts).all(), "FAIL: target distribution changed"

print("Output file verified.")
print(f"Distribution preserved: 0 = {saved_counts[0]:,}, 1 = {saved_counts[1]:,}")

Output file verified.
Distribution preserved: 0 = 17,208, 1 = 15,385


## Stage 3: Feature Engineering

In [48]:
WEEK_CUTOFFS = [2, 4, 6, 8]
DAYS_PER_WEEK = 7

KEYS = ["code_module", "code_presentation", "id_student"]


# Load Stage 1 outputs + Load VLE activity

In [50]:
def load_stage3_inputs():
    student_info_path = os.path.join(PROCESSED_DIR, "studentInfo_with_target.csv")

    if os.path.exists(student_info_path):
        student_info = pd.read_csv(student_info_path)
    else:
        raise FileNotFoundError(
            "Missing data/processed/studentInfo_with_target.csv. "
            "Run Stage 1 first so that the binary target is available."
        )

    student_vle = pd.read_csv(os.path.join(RAW_DIR, "studentVle.csv"))
    vle = pd.read_csv(os.path.join(RAW_DIR, "vle.csv"))

    return student_info, student_vle, vle

# Merge VLE metadata

In [51]:
def add_activity_type(student_vle, vle):
    vle_cols = [
        "id_site",
        "code_module",
        "code_presentation",
        "activity_type"
    ]

    return student_vle.merge(
        vle[vle_cols],
        on=["id_site", "code_module", "code_presentation"],
        how="left"
    )


# Aggregate VLE features

In [53]:
def aggregate_vle_until_cutoff(student_vle_enriched, cutoff_week):

    # Filter by cutoff week
    # Prevents future-data leakage
    cutoff_day = cutoff_week * DAYS_PER_WEEK

    vle_cut = student_vle_enriched[
        student_vle_enriched["date"] <= cutoff_day
    ].copy()


    # Aggregate total engagement
    # One row per student-course

    base_agg = vle_cut.groupby(KEYS).agg(
        total_clicks=("sum_click", "sum"),
        mean_daily_clicks=("sum_click", "mean"),
        max_daily_clicks=("sum_click", "max"),
        active_days=("date", "nunique"),
        unique_sites=("id_site", "nunique"),
        unique_activity_types=("activity_type", "nunique")
    ).reset_index()

    # Aggregate by activity type
    # Creates forum/resource/quiz click features
    activity_clicks = (
        vle_cut
        .pivot_table(
            index=KEYS,
            columns="activity_type",
            values="sum_click",
            aggfunc="sum",
            fill_value=0
        )
        .reset_index()
    )

    activity_clicks.columns = [
        col if col in KEYS else f"clicks_{str(col).lower().replace(' ', '_')}"
        for col in activity_clicks.columns
    ]

    # Combine VLE feature groups
    # Joins total and activity-type features

    features = base_agg.merge(activity_clicks, on=KEYS, how="left")

    # Create ratio features
    # Normalised engagement measures

    features["clicks_per_active_day"] = (
        features["total_clicks"] / features["active_days"].replace(0, np.nan)
    ).fillna(0)

    features["clicks_per_site"] = (
        features["total_clicks"] / features["unique_sites"].replace(0, np.nan)
    ).fillna(0)

    features["cutoff_week"] = cutoff_week
    features["cutoff_day"] = cutoff_day

    return features

# Merge with student demographics

In [56]:
def build_week_feature_table(student_info, vle_features, cutoff_week):
    demographic_cols = [
        "code_module",
        "code_presentation",
        "id_student",
        "gender",
        "region",
        "highest_education",
        "imd_band",
        "age_band",
        "num_of_prev_attempts",
        "studied_credits",
        "disability",
        "final_result",
        "target"
    ]

    df = student_info[demographic_cols].merge(
        vle_features,
        on=KEYS,
        how="left"
    )

    # Fill missing VLE values
    # Missing means no activity before cutoff
  
    engineered_cols = [
        col for col in df.columns
        if col not in demographic_cols
    ]

    df[engineered_cols] = df[engineered_cols].fillna(0)

    df["feature_set"] = f"week{cutoff_week}"

    return df


# Run Leakage check for forbidden future columns

In [57]:
def audit_no_leakage(df, cutoff_week):
    forbidden_columns = [
        "date",
        "date_submitted",
        "score",
        "date_unregistration"
    ]

    found = [col for col in forbidden_columns if col in df.columns]

    if found:
        raise ValueError(
            f"Leakage risk in week {cutoff_week}: forbidden columns found: {found}"
        )

    if "target" not in df.columns:
        raise ValueError(f"Week {cutoff_week}: missing binary target column.")

    if df["target"].isna().sum() > 0:
        raise ValueError(f"Week {cutoff_week}: target contains missing values.")

    return True


# Build feature tables

In [58]:
def save_features_table_manifest(feature_tables):
    rows = []

    for week, df in feature_tables.items():
        for col in df.columns:
            if col in KEYS:
                source = "studentInfo"
                definition = "Student-course identifier."
                computation = "Retained from Stage 1 studentInfo."
                leakage = "None"

            elif col in [
                "gender",
                "region",
                "highest_education",
                "imd_band",
                "age_band",
                "num_of_prev_attempts",
                "studied_credits",
                "disability"
            ]:
                source = "studentInfo"
                definition = "Student demographic or registration feature."
                computation = "Retained from Stage 1 studentInfo."
                leakage = "Low"

            elif col in ["final_result", "target"]:
                source = "studentInfo"
                definition = "Outcome label."
                computation = "Created in Stage 1 from final_result."
                leakage = "High if used as input feature."

            elif col.startswith("clicks_"):
                source = "studentVle + vle"
                definition = "Clicks for a specific VLE activity type."
                computation = "Sum of sum_click by student and activity_type."
                leakage = "None"

            elif col in [
                "total_clicks",
                "mean_daily_clicks",
                "max_daily_clicks",
                "active_days",
                "unique_sites",
                "unique_activity_types",
                "clicks_per_active_day",
                "clicks_per_site"
            ]:
                source = "studentVle"
                definition = "Aggregated VLE engagement before cutoff."
                computation = "Grouped aggregation up to cutoff week."
                leakage = "None"

            else:
                source = "engineered"
                definition = "Stage 3 helper or metadata feature."
                computation = "Created during feature engineering."
                leakage = "None"

            rows.append({
                "Week availability": week,
                "Feature": col,
                "Source CSV(s)": source,
                "Data type": str(df[col].dtype),
                "Definition": definition,
                "How computed": computation,
                "Missing count": int(df[col].isna().sum()),
                "Duplicate count": int(df.duplicated().sum()),
                "Leakage risk": leakage,
                "Notes": ""
            })

    manifest = pd.DataFrame(rows)

    # ------------------------------------------------------------
    # Save feature documentation
    # Records source, meaning, and leakage risk
    # ------------------------------------------------------------
    out_path = os.path.join(PROCESSED_DIR, "stage3_features_table.csv")
    manifest.to_csv(out_path, index=False)

    print(f"Saved feature manifest: {out_path}")


# Run all week cutoffs

In [59]:
def main():
    student_info, student_vle, vle = load_stage3_inputs()

    student_vle_enriched = add_activity_type(student_vle, vle)

    feature_tables = {}

    for week in WEEK_CUTOFFS:
        print(f"\nBuilding week {week} feature table...")

        vle_features = aggregate_vle_until_cutoff(
            student_vle_enriched,
            cutoff_week=week
        )

        week_df = build_week_feature_table(
            student_info,
            vle_features,
            cutoff_week=week
        )

        audit_no_leakage(week_df, cutoff_week=week)

        # ------------------------------------------------------------
        # Save week feature table
        # Output for Stages 4 and 5
        # ------------------------------------------------------------
        out_path = os.path.join(PROCESSED_DIR, f"week{week}_features.csv")
        week_df.to_csv(out_path, index=False)

        feature_tables[week] = week_df

        print(f"Saved: {out_path}")
        print(f"Shape: {week_df.shape[0]:,} rows x {week_df.shape[1]} columns")
        print("Target distribution:")
        print(week_df["target"].value_counts(normalize=True).round(3))

    save_features_table_manifest(feature_tables)

    print("\nStage 3 complete.")
    print("Generated week2, week4, week6, and week8 feature tables.")


if __name__ == "__main__":
    main()


Building week 2 feature table...
Saved: data\processed\week2_features.csv
Shape: 32,593 rows x 42 columns
Target distribution:
target
0    0.528
1    0.472
Name: proportion, dtype: float64

Building week 4 feature table...
Saved: data\processed\week4_features.csv
Shape: 32,593 rows x 42 columns
Target distribution:
target
0    0.528
1    0.472
Name: proportion, dtype: float64

Building week 6 feature table...
Saved: data\processed\week6_features.csv
Shape: 32,593 rows x 42 columns
Target distribution:
target
0    0.528
1    0.472
Name: proportion, dtype: float64

Building week 8 feature table...
Saved: data\processed\week8_features.csv
Shape: 32,593 rows x 43 columns
Target distribution:
target
0    0.528
1    0.472
Name: proportion, dtype: float64
Saved feature manifest: data\processed\stage3_features_table.csv

Stage 3 complete.
Generated week2, week4, week6, and week8 feature tables.


In [60]:
for week in [2, 4, 6, 8]:
    path = os.path.join(PROCESSED_DIR, f"week{week}_features.csv")
    df = pd.read_csv(path)

    print("\n" + "="*80)
    print(f"WEEK {week} FEATURE TABLE")
    print("="*80)

    print(f"Shape: {df.shape[0]} rows x {df.shape[1]} columns\n")

    print("First 5 rows:")
    display(df.head())

    print("\nColumns:")
    print(df.columns.tolist())


WEEK 2 FEATURE TABLE
Shape: 32593 rows x 42 columns

First 5 rows:


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,clicks_quiz,clicks_resource,clicks_sharedsubpage,clicks_subpage,clicks_url,clicks_per_active_day,clicks_per_site,cutoff_week,cutoff_day,feature_set
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,8.0,0.0,20.0,1.0,50.166667,14.333333,2.0,14.0,week2
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,5.0,0.0,43.0,19.0,32.571429,15.724138,2.0,14.0,week2
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,4.0,0.0,22.0,4.0,23.416667,12.772727,2.0,14.0,week2
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,10.0,0.0,38.0,13.0,26.846154,14.541667,2.0,14.0,week2
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,7.0,0.0,20.0,3.0,27.764706,15.733333,2.0,14.0,week2



Columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'target', 'total_clicks', 'mean_daily_clicks', 'max_daily_clicks', 'active_days', 'unique_sites', 'unique_activity_types', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url', 'clicks_per_active_day', 'clicks_per_site', 'cutoff_week', 'cutoff_day', 'feature_set']

WEEK 4 FEATURE TABLE
Shape: 32593 rows x 42 columns

First 5 rows:


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,clicks_quiz,clicks_resource,clicks_sharedsubpage,clicks_subpage,clicks_url,clicks_per_active_day,clicks_per_site,cutoff_week,cutoff_day,feature_set
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,9.0,0.0,21.0,1.0,50.125000,16.708333,4.0,28.0,week4
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,5.0,0.0,59.0,30.0,32.526316,18.176471,4.0,28.0,week4
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,4.0,0.0,22.0,4.0,23.416667,12.772727,4.0,28.0,week4
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,10.0,0.0,52.0,17.0,22.454545,15.935484,4.0,28.0,week4
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,7.0,0.0,28.0,6.0,23.625000,16.676471,4.0,28.0,week4



Columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'target', 'total_clicks', 'mean_daily_clicks', 'max_daily_clicks', 'active_days', 'unique_sites', 'unique_activity_types', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url', 'clicks_per_active_day', 'clicks_per_site', 'cutoff_week', 'cutoff_day', 'feature_set']

WEEK 6 FEATURE TABLE
Shape: 32593 rows x 42 columns

First 5 rows:


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,clicks_quiz,clicks_resource,clicks_sharedsubpage,clicks_subpage,clicks_url,clicks_per_active_day,clicks_per_site,cutoff_week,cutoff_day,feature_set
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,9.0,0.0,23.0,1.0,37.461538,15.218750,6.0,42.0,week6
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,5.0,0.0,60.0,31.0,28.409091,18.382353,6.0,42.0,week6
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,4.0,0.0,22.0,4.0,23.416667,12.772727,6.0,42.0,week6
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,10.0,0.0,66.0,28.0,21.406250,19.027778,6.0,42.0,week6
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,7.0,0.0,28.0,6.0,23.115385,17.171429,6.0,42.0,week6



Columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'target', 'total_clicks', 'mean_daily_clicks', 'max_daily_clicks', 'active_days', 'unique_sites', 'unique_activity_types', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url', 'clicks_per_active_day', 'clicks_per_site', 'cutoff_week', 'cutoff_day', 'feature_set']

WEEK 8 FEATURE TABLE
Shape: 32593 rows x 43 columns

First 5 rows:


,code_module,code_presentation,id_student,gender,region,highest_education,imd_band,age_band,num_of_prev_attempts,studied_credits,...,clicks_repeatactivity,clicks_resource,clicks_sharedsubpage,clicks_subpage,clicks_url,clicks_per_active_day,clicks_per_site,cutoff_week,cutoff_day,feature_set
0,AAA,2013J,11391,M,East Anglian Region,HE Qualification,90-100%,55<=,0,240,...,0.0,9.0,0.0,23.0,1.0,29.388889,16.030303,8.0,56.0,week8
1,AAA,2013J,28400,F,Scotland,HE Qualification,20-30%,35-55,0,60,...,0.0,5.0,0.0,61.0,32.0,25.730769,19.676471,8.0,56.0,week8
2,AAA,2013J,30268,F,North Western Region,A Level or Equivalent,30-40%,35-55,0,60,...,0.0,4.0,0.0,22.0,4.0,23.416667,12.772727,8.0,56.0,week8
3,AAA,2013J,31604,F,South East Region,A Level or Equivalent,50-60%,35-55,0,60,...,0.0,10.0,0.0,75.0,36.0,20.100000,21.729730,8.0,56.0,week8
4,AAA,2013J,32885,F,West Midlands Region,Lower Than A Level,50-60%,0-35,0,60,...,0.0,7.0,0.0,28.0,6.0,23.115385,17.171429,8.0,56.0,week8



Columns:
['code_module', 'code_presentation', 'id_student', 'gender', 'region', 'highest_education', 'imd_band', 'age_band', 'num_of_prev_attempts', 'studied_credits', 'disability', 'final_result', 'target', 'total_clicks', 'mean_daily_clicks', 'max_daily_clicks', 'active_days', 'unique_sites', 'unique_activity_types', 'clicks_dataplus', 'clicks_dualpane', 'clicks_externalquiz', 'clicks_forumng', 'clicks_glossary', 'clicks_homepage', 'clicks_htmlactivity', 'clicks_oucollaborate', 'clicks_oucontent', 'clicks_ouelluminate', 'clicks_ouwiki', 'clicks_page', 'clicks_questionnaire', 'clicks_quiz', 'clicks_repeatactivity', 'clicks_resource', 'clicks_sharedsubpage', 'clicks_subpage', 'clicks_url', 'clicks_per_active_day', 'clicks_per_site', 'cutoff_week', 'cutoff_day', 'feature_set']
